In [ ]:
%run bootstrap.ipynb

# THE VACUUM WORLD   

In this notebook, we will be discussing **the structure of agents** through an example of the **vacuum agent**. The job of AI is to design an **agent program** that implements the agent function: the mapping from percepts to actions. We assume this program will run on some sort of computing device with physical sensors and actuators: we call this the **architecture**:

<h3 align="center">agent = architecture + program</h3>

## Setup

In [ ]:
# Imports
from aima.agents import *
from aima.notebook_utils import psource

In [ ]:
# Helper class/functions for the 2x2 environment
class DirectionType(type):
    """Fancy python trickery that allows us to use either
        >> Direction.Right
        or
        >> Direction["Right"]
    and get the same outcome (1,0)
    """
    Right = (1,0)
    Left  = (-1,0)
    Down  = (0,-1)
    Up    = (0,1)
    
    def __getitem__(self, key):
        match key.lower():
            case "right": return self.Right
            case "left": return self.Left
            case "down": return self.Down
            case "up": return self.Up
            case _: return (0,0) # default case

class Directions(metaclass=DirectionType):
    """A simple helper class for moving around a 2d grid."""
    pass

all_literal_directions = ["Right","Left","Up","Down"]

def add_direction(position, direction):
    if direction in all_literal_directions:
        direction_offset = Directions[direction]
    else:
        direction_offset = direction
    return (position[0] + direction_offset[0], position[1] + direction_offset[1])

The actual 2x2 environment

In [ ]:
grid2x2 = [
    (x,y)
    for x in range(2)
    for y in range(2)
]

class VacuumEnvironment2d(TrivialVacuumEnvironment):

    def __init__(self):
        super().__init__()
        # We're overriding the status they set here
        self.status = {
            location: random.choice(['Clean', 'Dirty'])
            for location in grid2x2
        }

    def execute_action(self, agent, action):
        """Change agent's location and/or location's status; track performance.
        Score 10 for each dirt cleaned; -1 for each move."""
        location_x, location_y = agent.location
        
        if action in all_literal_directions:
            agent.performance -= 1
            destination = add_direction(agent.location, Directions[action])
            # Only move if it is to a location inside the grid
            if destination in self.status:
                agent.location = destination
        elif action == "Suck":
            if self.status[agent.location] == 'Dirty':
                agent.performance += 10
                self.status[agent.location] = 'Clean'
                

    def default_location(self, thing):
        """Agents start in either location at random."""
        all_locations = list(self.status.keys())
        return random.choice(all_locations)

## Q1 - Random Agent Program

In [ ]:
e = VacuumEnvironment2d()
a = Agent(program=RandomAgentProgram(['Right', 'Left', 'Up', 'Down', 'Suck', 'NoOp']))
e.add_thing(a)
print(e.status)
e.run(20)
print(e.status)
print(a.performance)

## Q2 - SIMPLE REFLEX AGENT PROGRAM

In [ ]:
def SimpleReflexAgentProgram2d():
    """This agent takes action based solely on the percept. [Figure 2.10]"""
    
    def program(percept):
        loc, status = percept
        if status == "Dirty":
            return "Suck"
        
        match loc:
            case (0,0): return "Right"
            case (1,0): return "Up"
            case (1,1): return "Left"
            case (0,1): return "Down"

    return program

In [ ]:
new_2d_env = VacuumEnvironment2d()

program = SimpleReflexAgentProgram2d()
simple_reflex_agent = Agent(program)

new_2d_env.add_thing(simple_reflex_agent)

print("State of the Environment Before: {}.".format(new_2d_env.status))

new_2d_env.run(20)

print("State of the Environment After:  {}.".format(new_2d_env.status))

print("ModelBasedVacuumAgent scored {}.".format(simple_reflex_agent.performance))

## Q3 - TABLE-DRIVEN AGENT PROGRAM

In [ ]:
#here I'll implement a function that can build a table to the full coverage depth for a 2x2 environment
#cumulative growth in rows of options per percept
#3 moves + 4 suck actions = 7 percepts
#Depth 7 costs 8 + 12 + 20 + 32 + 52 + 84 + 136 = 344
#number of rows per depth grows by the successive difference of the last size
#vacuum percept has 2 actions (tuple), move and suck
#those care about where the vacuum is and if the floor is dirty
#location, status = percept
#what are the locations?
#where does the agent move to
#when the agent is at location x where does it go next?

#location list variable
#move list variable
#move-next list variable

loc_A, loc_B, loc_C, loc_D = (0, 0), (1, 0), (1, 1), (0, 1)

LOC = [loc_A, loc_B, loc_C, loc_D]
MOVE = {loc_A:'Right', loc_B:'Up', loc_C:'Left', loc_D:'Down'}
NEXT = {loc_A: loc_B, loc_B: loc_C, loc_C: loc_D, loc_D: loc_A}

#I want the agent to suck up dirty loc & move past clean spots
#thats the perception turned into an action
def actionPerc(percept):
  location, status = percept

  if status == 'Dirty':
    return 'Suck'
  else:
    return MOVE[location]

#I want the agent to branch from one percept to the next possible percepts
def nextPerc(percept, action):
  location, status = percept

  if action == 'Suck':
    return [(location, 'Clean')]
  else:
    return [(NEXT[location], 'Dirty'), (NEXT[location], 'Clean')]

#table generation process:
#percept -> history -> action from newest percept
# -> next percept -> repeat

#depth = 2n-1
depth = 2*len(LOC)-1

#table trajectory:
#(A,Dirty):'Suck' -> (A, Clean):'Right' -> branch ->
 # (B,Dirty):'Suck' & (B,Clean):'Up'
def buildTable(depth):
  table = {}
  histories = []

  #for each location in the grid space...
  for location in LOC:
    for status in ['Clean','Dirty']:
    #create percept
      percept = (location, status)
    #create history
      history = (percept,)
    #create a starting history
      histories.append(history)

    #determine action associated w/ history


    #add history/action to table
  for i in range(depth):
    nextHist = []

    for history in histories:
      percept = history[-1]
      action = actionPerc(percept)
      table[history] = action

      for newPerc in nextPerc(percept, action):
        nextHist.append(history + (newPerc,))

    histories = nextHist


  return table

In [ ]:
e = VacuumEnvironment2d()
a = Agent(program=TableDrivenAgentProgram(table=buildTable(7)))
e.add_thing(a)
print(e.status)
e.run(20)
print(e.status)
print(a.performance)

## MODEL-BASED REFLEX AGENT PROGRAM

In [ ]:
def ModelBasedVacuumAgent2d():
    model = {
        location: None
        for location in grid2x2
    }

    def program(percept):
        """Same as ReflexVacuumAgent, except if everything is clean, do NoOp."""
        location, status = percept
        model[location] = status
        
        if status == "Dirty":
            return "Suck"
        
        adjacent_locations = []
        for direction in all_literal_directions:
            adjacent_spot = add_direction(location, Directions[direction])
            if adjacent_spot in model and model[adjacent_spot] != "Clean":
                adjacent_locations.append(direction)
        
        if len(adjacent_locations) == 0:
            return "NoOp"
        else:
            return adjacent_locations[0]
        

    return Agent(program)

In [ ]:
new_2d_env = VacuumEnvironment2d()

model_based_vacuum_agent_2d = ModelBasedVacuumAgent2d()

new_2d_env.add_thing(model_based_vacuum_agent_2d)

print("State of the Environment Before: {}.".format(new_2d_env.status))

new_2d_env.run()

print("State of the Environment After:  {}.".format(new_2d_env.status))

print("ModelBasedVacuumAgent scored {}.".format(model_based_vacuum_agent_2d.performance))